# Dopamine PINN (2D) — Complete Colab Pipeline

Companion notebook for paper **B3**: *Physics-Informed Neural Networks for Modeling Dopamine Neurotransmitter Diffusion in Synaptic Clefts* (PLOS Computational Biology).

**This is the 2D version.** The PDE is now
$$\partial_t C = D\,\nabla^2 C - k C, \quad (x, y) \in (-L/2, L/2)^2,\ t \in (0, T]$$
with radially symmetric Gaussian initial condition and zero-flux Neumann boundary conditions on all four edges of the square domain.

## What this notebook does

1. Forward 2D PINN: solve the planar dopamine reaction-diffusion PDE, compare against the 2D analytical solution and a 2D finite-difference reference solver
2. Inverse 2D PINN: recover $D$ and $k$ from noisy synthetic observations sampled over $(x, y, t)$
3. Auto-fill the LaTeX manuscript with the computed numbers
4. Download all artifacts

## Runtime

**Select a GPU runtime**: *Runtime → Change runtime type → GPU* (T4 is sufficient). Expect 10–20 minutes total at full fidelity.

## Toggle flags (in Cell 4)

| Flag | Effect |
|---|---|
| `QUICK = True` | Reduces iterations / collocation points for ~5-minute sanity check |
| `AUTO_FILL = True` | Prompts for `B3_Dopamine_PINN_Paper.tex` upload and substitutes the [TBD] markers |

## 0. Install dependencies and set PyTorch backend

In [ ]:
!pip -q install deepxde scipy matplotlib
%env DDE_BACKEND=pytorch

In [ ]:
import os, time, json, re
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import deepxde as dde
from deepxde.backend import torch
from scipy.interpolate import RegularGridInterpolator

SEED = 1234
np.random.seed(SEED)
dde.config.set_random_seed(SEED)

FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)
print('DeepXDE backend:', dde.backend.backend_name)
print('CUDA available:', torch.cuda.is_available())

## 1. Toggles and physical parameters

In [ ]:
# Toggles
QUICK     = False    # ~5-minute sanity check
AUTO_FILL = False    # upload B3.tex, substitute, download

# Physical parameters
D_TRUE = 0.32     # diffusion coefficient (mu_m^2 / ms)
K_TRUE = 0.05     # linear reuptake rate (1 / ms)
L      = 5.0      # domain side length (mu_m); domain is [-L/2, L/2]^2
T      = 20.0     # simulation time (ms)
SIGMA  = 0.5      # release pulse width (mu_m)
C0     = 1.0      # peak release concentration (mu_M)

# Network and training hyperparameters
N_DOMAIN    = 10_000
N_BOUNDARY  = 400         # 100 per edge x 4 edges
N_INITIAL   = 400         # over the 2D square
N_TEST      = 5_000
LAYERS      = [3] + [64] * 4 + [1]   # input is (x, y, t)
ACTIVATION  = 'tanh'
INITIALIZER = 'Glorot normal'
ADAM_ITERS  = 20_000
LR          = 1e-3

# Inverse-problem observation count.
# 1D used N_OBS = 100. The 2D analogue at the same per-area density
# requires 400 observations (4x scaling from a 1D segment to a 2D
# square at fixed sampling resolution). Sparser sampling in 2D
# produces underdetermined inverse problems with biased recovery.
N_OBS = 400

if QUICK:
    N_DOMAIN, N_BOUNDARY, N_INITIAL, N_TEST = 1_000, 80, 80, 500
    ADAM_ITERS = 1_000
    N_OBS = 80
    print('[QUICK MODE] Reduced hyperparameters - not paper-quality.')

print(f'D = {D_TRUE} mu_m^2/ms, k = {K_TRUE} 1/ms')
print(f'Domain: [-{L/2}, {L/2}]^2 mu_m, T = {T} ms')
print(f'Architecture: {LAYERS}, Adam iters: {ADAM_ITERS}, N_OBS = {N_OBS}')

## 2. Analytical solution (2D)

Closed-form for an infinite planar domain (see Appendix B):

$$C(x, y, t) = \frac{C_0\,\sigma^2}{\sigma^2 + 2Dt} \exp\!\left(-\frac{x^2 + y^2}{2(\sigma^2 + 2Dt)}\right) e^{-kt}$$

Note the amplitude factor is $\sigma^2/(\sigma^2 + 2Dt)$ in 2D (vs. $\sigma/\sqrt{\sigma^2 + 2Dt}$ in 1D).

In [ ]:
def C_analytical(x, y, t, D=D_TRUE, k=K_TRUE, sigma=SIGMA, C0=C0):
    s2 = sigma ** 2 + 2.0 * D * t
    return C0 * sigma**2 / s2 * np.exp(-(x ** 2 + y ** 2) / (2.0 * s2)) * np.exp(-k * t)

xs = np.linspace(-L/2, L/2, 81)
ys = np.linspace(-L/2, L/2, 81)
Xg, Yg = np.meshgrid(xs, ys, indexing='ij')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, t_snap in zip(axes, [0, 5, 10]):
    C = C_analytical(Xg, Yg, t_snap)
    im = ax.imshow(C.T, extent=[-L/2, L/2, -L/2, L/2], origin='lower', cmap='viridis')
    ax.set_xlabel('x (mu_m)'); ax.set_ylabel('y (mu_m)')
    ax.set_title(f't = {t_snap} ms')
    plt.colorbar(im, ax=ax)
fig.suptitle('Analytical solution (2D Gaussian spreading + reuptake)')
fig.tight_layout(); plt.show()

## 3. Finite-difference reference solver (2D)

Explicit-time 5-point Laplacian stencil with Neumann (zero-flux) boundaries on all four edges. Stability condition in 2D: $\Delta t \le 1/(2D(\Delta x^{-2} + \Delta y^{-2}))$.

In [ ]:
def fd_reference(D=D_TRUE, k=K_TRUE, L=L, T=T, sigma=SIGMA, C0=C0,
                 nx=81, ny=81, nt=None):
    dx = L / (nx - 1)
    dy = L / (ny - 1)
    if nt is None:
        # 2D stability: dt <= 1 / (2D * (1/dx^2 + 1/dy^2)); 5% safety
        dt_max = 0.95 / (2.0 * D * (1.0/dx**2 + 1.0/dy**2))
        nt = int(np.ceil(T / dt_max)) + 1
    dt = T / (nt - 1)
    assert dt <= 1.0 / (2.0 * D * (1.0/dx**2 + 1.0/dy**2)), 'FD stability violated.'
    x = np.linspace(-L/2, L/2, nx)
    y = np.linspace(-L/2, L/2, ny)
    Xg, Yg = np.meshgrid(x, y, indexing='ij')
    C = C0 * np.exp(-(Xg**2 + Yg**2) / (2.0 * sigma**2))
    history = np.zeros((nt, nx, ny))
    history[0] = C
    print(f'FD: nx={nx}, ny={ny}, nt={nt}, dt={dt:.4e} ms')
    for n in range(1, nt):
        lap = np.zeros_like(C)
        # d^2/dx^2 with Neumann mirror at x = +-L/2
        lap[1:-1, :] += (C[2:, :] - 2*C[1:-1, :] + C[:-2, :]) / dx**2
        lap[0,   :]  += 2 * (C[1,  :] - C[0,  :]) / dx**2
        lap[-1,  :]  += 2 * (C[-2, :] - C[-1, :]) / dx**2
        # d^2/dy^2 with Neumann mirror at y = +-L/2
        lap[:, 1:-1] += (C[:, 2:] - 2*C[:, 1:-1] + C[:, :-2]) / dy**2
        lap[:, 0]    += 2 * (C[:, 1]  - C[:, 0])  / dy**2
        lap[:, -1]   += 2 * (C[:, -2] - C[:, -1]) / dy**2
        C = C + D * dt * lap - k * dt * C
        history[n] = C
    return x, y, np.linspace(0, T, nt), history

t0 = time.time()
x_fd, y_fd, t_fd, C_fd_full = fd_reference()
print(f'FD run: {time.time()-t0:.1f} s, shape={C_fd_full.shape}, max C={C_fd_full.max():.4f} mu_M')

## 4. Forward PINN (2D)

Network input is $(x, y, t) \in \mathbb{R}^3$. PDE residual uses both spatial second derivatives:
$$\mathcal{L}_r(\theta) = \big| \partial_t \hat{C} - D(\partial_{xx} + \partial_{yy}) \hat{C} + k \hat{C} \big|^2$$

In [ ]:
geom     = dde.geometry.Rectangle([-L/2, -L/2], [L/2, L/2])
timedom  = dde.geometry.TimeDomain(0, T)
geomtime = dde.geometry.GeometryXTime(geom, timedom)

def pde_forward(X, C):
    # X columns: [x, y, t]
    dC_t  = dde.grad.jacobian(C, X, i=0, j=2)
    dC_xx = dde.grad.hessian(C, X, i=0, j=0)
    dC_yy = dde.grad.hessian(C, X, i=1, j=1)
    return dC_t - D_TRUE * (dC_xx + dC_yy) + K_TRUE * C

def initial_condition(X):
    x = X[:, 0:1]
    y = X[:, 1:2]
    return C0 * np.exp(-(x**2 + y**2) / (2.0 * SIGMA**2))

ic = dde.icbc.IC(geomtime, initial_condition, lambda _, on_initial: on_initial)
bc = dde.icbc.NeumannBC(geomtime,
                        lambda X: np.zeros((len(X), 1)),
                        lambda _, on_boundary: on_boundary)

def build_forward(layers=LAYERS, n_domain=N_DOMAIN):
    data = dde.data.TimePDE(
        geomtime, pde_forward, [ic, bc],
        num_domain=n_domain, num_boundary=N_BOUNDARY,
        num_initial=N_INITIAL, num_test=N_TEST,
    )
    net = dde.nn.FNN(layers, ACTIVATION, INITIALIZER)
    return dde.Model(data, net)

model_fwd = build_forward()

In [ ]:
t0 = time.time()
model_fwd.compile('adam', lr=LR)
model_fwd.train(iterations=ADAM_ITERS, display_every=2_000)
model_fwd.compile('L-BFGS')
model_fwd.train()
print(f'Forward training time: {time.time() - t0:.1f} s')

### 4.1 Forward-problem evaluation

Evaluate the PINN on a 3D space-time grid; compute L2 errors vs. analytical and FD references; report per-snapshot errors.

In [ ]:
# Evaluation grid: 41 x 41 spatial x 21 time slices
nx_eval, ny_eval, nt_eval = 41, 41, 21
xs = np.linspace(-L/2, L/2, nx_eval)
ys = np.linspace(-L/2, L/2, ny_eval)
ts = np.linspace(0, T, nt_eval)
Xg, Yg, Tg = np.meshgrid(xs, ys, ts, indexing='ij')
XYT = np.stack([Xg.ravel(), Yg.ravel(), Tg.ravel()], axis=1)

C_pinn  = model_fwd.predict(XYT).reshape(nx_eval, ny_eval, nt_eval)
C_exact = C_analytical(Xg, Yg, Tg)
interp  = RegularGridInterpolator((t_fd, x_fd, y_fd), C_fd_full,
                                  bounds_error=False, fill_value=0.0)
C_fd    = interp(np.stack([Tg.ravel(), Xg.ravel(), Yg.ravel()], axis=1)
                ).reshape(nx_eval, ny_eval, nt_eval)

err_anal = 100.0 * np.linalg.norm(C_pinn - C_exact) / np.linalg.norm(C_exact)
err_fd   = 100.0 * np.linalg.norm(C_pinn - C_fd) / np.linalg.norm(C_fd)
print(f'L2(PINN vs. analytical, full window) = {err_anal:.3f}%')
print(f'L2(PINN vs. FD reference, full window) = {err_fd:.3f}%')

per_snapshot = {}
for t_snap in (1.0, 5.0, 10.0, 20.0):
    i_t = np.argmin(np.abs(ts - t_snap))
    C_p = C_pinn[:, :, i_t]
    C_a = C_exact[:, :, i_t]
    C_f = C_fd[:, :, i_t]
    per_snapshot[f't_{int(t_snap)}'] = {
        'L2_anal_pct': 100.0 * np.linalg.norm(C_p - C_a) / np.linalg.norm(C_a),
        'L2_fd_pct':   100.0 * np.linalg.norm(C_p - C_f) / np.linalg.norm(C_f),
    }
    e = per_snapshot[f't_{int(t_snap)}']
    print(f'  t = {t_snap:5.1f} ms:  vs. analytical = {e["L2_anal_pct"]:6.2f}%   vs. FD = {e["L2_fd_pct"]:6.2f}%')

In [ ]:
# Snapshot panel: heatmaps of C(x, y) at three time slices, all showing the PINN prediction
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
vmax = C_pinn.max()
for ax, t_snap in zip(axes, [0.0, T/4, T/2]):
    i_t = np.argmin(np.abs(ts - t_snap))
    im = ax.imshow(C_pinn[:, :, i_t].T, extent=[-L/2, L/2, -L/2, L/2],
                   origin='lower', cmap='viridis', vmin=0, vmax=vmax)
    ax.set_xlabel('x (mu_m)'); ax.set_ylabel('y (mu_m)')
    ax.set_title(f't = {t_snap:.1f} ms')
    plt.colorbar(im, ax=ax, label='C (mu_M)')
fig.suptitle('Forward problem: PINN concentration field C(x, y, t)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'forward_snapshots.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Comparison panel at t = T/4: analytical, PINN, |error|
i_t = np.argmin(np.abs(ts - T/4))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, field, title in zip(
    axes,
    [C_exact[:, :, i_t], C_pinn[:, :, i_t], np.abs(C_exact[:, :, i_t] - C_pinn[:, :, i_t])],
    ['Analytical', 'PINN', '|Analytical - PINN|'],
):
    im = ax.imshow(field.T, extent=[-L/2, L/2, -L/2, L/2],
                   origin='lower', cmap='viridis')
    ax.set_xlabel('x (mu_m)'); ax.set_ylabel('y (mu_m)')
    ax.set_title(title)
    plt.colorbar(im, ax=ax)
fig.suptitle(f'C(x, y) at t = {T/4:.1f} ms')
fig.tight_layout()
fig.savefig(FIG_DIR / 'forward_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

## 5. Inverse PINN — recover $D$ and $k$ from noisy observations

100 observations sampled uniformly over $(x, y) \in \Omega$ and $t \in [0.5, T/2]$, with 2% Gaussian noise. Clean concentrations come from the FD reference (consistent with the PINN's Neumann BCs). Trainable parameters are log-parameterized to enforce positivity; residual loss is upweighted $10\times$ vs. the data loss.

In [ ]:
def make_noisy_observations(n_obs=N_OBS, noise_pct=2.0, rng=None):
    rng = rng or np.random.default_rng(SEED)
    obs_x = rng.uniform(-L/2, L/2, n_obs)
    obs_y = rng.uniform(-L/2, L/2, n_obs)
    obs_t = rng.uniform(0.5, T/2.0, n_obs)
    interp_fd = RegularGridInterpolator((t_fd, x_fd, y_fd), C_fd_full,
                                        bounds_error=False, fill_value=0.0)
    C_clean = interp_fd(np.stack([obs_t, obs_x, obs_y], axis=1))
    obs_C = C_clean + rng.normal(0.0, (noise_pct / 100.0) * C0, n_obs)
    return obs_x, obs_y, obs_t, obs_C

obs_x, obs_y, obs_t, obs_C = make_noisy_observations(n_obs=N_OBS, noise_pct=2.0)

fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(obs_x, obs_y, c=obs_t, cmap='plasma', s=20, alpha=0.7)
ax.set_xlabel('x (mu_m)'); ax.set_ylabel('y (mu_m)')
ax.set_xlim(-L/2, L/2); ax.set_ylim(-L/2, L/2)
ax.set_title(f'{len(obs_x)} noisy observations (color = observation time, 2% noise)')
plt.colorbar(sc, ax=ax, label='t (ms)')
plt.show()

In [ ]:
# Log-parametrization: D = exp(D_log), k = exp(k_log) > 0 strictly
# Initial guesses close to (but not equal to) the truth.
def train_inverse(obs_x, obs_y, obs_t, obs_C):
    D_log = dde.Variable(float(np.log(0.30)))
    k_log = dde.Variable(float(np.log(0.04)))

    def pde_inverse(X, C):
        D = torch.exp(D_log)
        k = torch.exp(k_log)
        dC_t  = dde.grad.jacobian(C, X, i=0, j=2)
        dC_xx = dde.grad.hessian(C, X, i=0, j=0)
        dC_yy = dde.grad.hessian(C, X, i=1, j=1)
        return dC_t - D * (dC_xx + dC_yy) + k * C

    obs_coords = np.stack([obs_x, obs_y, obs_t], axis=1)
    observe = dde.icbc.PointSetBC(obs_coords, obs_C.reshape(-1, 1), component=0)

    data = dde.data.TimePDE(
        geomtime, pde_inverse, [ic, bc, observe],
        num_domain=N_DOMAIN, num_boundary=N_BOUNDARY,
        num_initial=N_INITIAL, num_test=N_TEST,
        anchors=obs_coords,
    )
    net = dde.nn.FNN(LAYERS, ACTIVATION, INITIALIZER)
    model = dde.Model(data, net)

    var_cb = dde.callbacks.VariableValue(
        [D_log, k_log], period=500, filename=str(FIG_DIR / 'variables.dat'),
    )

    loss_weights = [10.0, 1.0, 1.0, 1.0]   # [residual, IC, BC, data]
    t0 = time.time()
    model.compile('adam', lr=LR, loss_weights=loss_weights,
                  external_trainable_variables=[D_log, k_log])
    model.train(iterations=ADAM_ITERS, display_every=2_000, callbacks=[var_cb])
    model.compile('L-BFGS', loss_weights=loss_weights,
                  external_trainable_variables=[D_log, k_log])
    model.train(callbacks=[var_cb])
    print(f'Inverse training time: {time.time() - t0:.1f} s')

    D_rec = float(torch.exp(D_log).detach().cpu().numpy())
    k_rec = float(torch.exp(k_log).detach().cpu().numpy())
    return model, D_rec, k_rec

model_inv, D_rec, k_rec = train_inverse(obs_x, obs_y, obs_t, obs_C)
rel_D = 100.0 * abs(D_rec - D_TRUE) / D_TRUE
rel_k = 100.0 * abs(k_rec - K_TRUE) / K_TRUE
print()
print(f'Recovered D = {D_rec:.4f} mu_m^2/ms (true 0.32, |err| = {rel_D:.2f}%)')
print(f'Recovered k = {k_rec:.4f} 1/ms      (true 0.05, |err| = {rel_k:.2f}%)')

In [ ]:
# Convergence plot: parse DeepXDE's bracketed variables.dat and apply exp() to recover D, k
iters, Ds, ks = [], [], []
with open(FIG_DIR / 'variables.dat') as fh:
    for line in fh:
        parts = line.strip().split(None, 1)
        if len(parts) != 2:
            continue
        try:
            it = int(parts[0])
        except ValueError:
            continue
        nums = re.findall(r'[+-]?\d+\.?\d*(?:[eE][+-]?\d+)?', parts[1])
        if len(nums) >= 2:
            iters.append(it)
            Ds.append(np.exp(float(nums[0])))
            ks.append(np.exp(float(nums[1])))
iters, Ds, ks = np.array(iters), np.array(Ds), np.array(ks)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(iters, Ds, 'r-')
axes[0].axhline(D_TRUE, color='k', ls=':', label='true')
axes[0].set_xlabel('iteration'); axes[0].set_ylabel('D (mu_m^2/ms)')
axes[0].set_title('Diffusion coefficient recovery'); axes[0].legend()
axes[1].plot(iters, ks, 'b-')
axes[1].axhline(K_TRUE, color='k', ls=':', label='true')
axes[1].set_xlabel('iteration'); axes[1].set_ylabel('k (1/ms)')
axes[1].set_title('Reuptake rate recovery'); axes[1].legend()
fig.tight_layout()
fig.savefig(FIG_DIR / 'inverse_convergence.png', dpi=200, bbox_inches='tight')
plt.show()

## 6. Save `metrics.json` and auto-fill the manuscript

If `AUTO_FILL = True` in Cell 4, the next two cells:
1. Save the computed metrics to `figures/metrics.json`
2. Prompt you to upload `B3_Dopamine_PINN_Paper.tex`
3. Replace the `[TBD]` markers with the actual computed values
4. Download the filled `.tex`

In [ ]:
metrics = {
    'dimension':       '2D',
    'true_params':     {'D': D_TRUE, 'k': K_TRUE},
    'hyperparameters': {
        'layers': LAYERS, 'activation': ACTIVATION,
        'adam_iters': ADAM_ITERS, 'n_domain': N_DOMAIN,
        'noise_pct': 2.0, 'quick_mode': QUICK,
    },
    'forward': {
        'L2_vs_analytical_pct': float(err_anal),
        'L2_vs_fd_pct':         float(err_fd),
        'per_snapshot':         per_snapshot,
    },
    'inverse': {
        'D_recovered':     D_rec, 'D_rel_error_pct': float(rel_D),
        'k_recovered':     k_rec, 'k_rel_error_pct': float(rel_k),
        'n_observations':  int(N_OBS), 'noise_pct': 2.0,
    },
}
with open(FIG_DIR / 'metrics.json', 'w') as fh:
    json.dump(metrics, fh, indent=2)
print(f'Saved {FIG_DIR / "metrics.json"}')
print()
print(json.dumps(metrics, indent=2))

In [ ]:
if AUTO_FILL:
    try:
        from google.colab import files
        print('Please select B3_Dopamine_PINN_Paper.tex to upload...')
        uploaded = files.upload()
        tex_name = next(iter(uploaded.keys()))
    except ImportError:
        tex_name = 'B3_Dopamine_PINN_Paper.tex'
        if not Path(tex_name).exists():
            raise FileNotFoundError(f'{tex_name} not found in working directory.')

    text = Path(tex_name).read_text()

    # Use t = 1 ms per-snapshot L2 vs analytical as the headline
    # (consistent with the manuscript phrasing).
    L2_anal_headline = per_snapshot['t_1']['L2_anal_pct']
    max_rec_err = max(rel_D, rel_k)

    # Replacement order matters: do the precise patterns first.
    replacements = [
        # Forward L2 vs FD (full window) -- the headline accuracy claim
        (r'L2 relative error of \\textbf\{\[TBD\]\}\\% compared with a finite-difference',
         f'L2 relative error of {err_fd:.2f}\\% compared with a finite-difference'),
        # Forward L2 vs analytical (t = 1 ms)
        (r'L2 relative error of \\textbf\{\[TBD\]\}\\% compared with\s+the analytical',
         f'L2 relative error of {L2_anal_headline:.2f}\\% compared with the analytical'),
        # Conclusion: L2 relative error of [TBD]% against the analytical
        (r'L2 relative error of \\textbf\{\[TBD\]\}\\% against the analytical',
         f'L2 relative error of {L2_anal_headline:.2f}\\% against the analytical'),
        # Recovered D, k values
        (r'\$D =\$ \\textbf\{\[TBD\]\}', f'$D =$ {D_rec:.4f}'),
        (r'\$k =\$ \\textbf\{\[TBD\]\}', f'$k =$ {k_rec:.4f}'),
        # Abstract: relative errors of [TBD]% and [TBD]%
        (r'relative errors of \\textbf\{\[TBD\]\}\\% and \\textbf\{\[TBD\]\}\\%',
         f'relative errors of {rel_D:.2f}\\% and {rel_k:.2f}\\%'),
        # Conclusion: relative errors below [TBD]%
        (r'with relative\s+errors below \\textbf\{\[TBD\]\}\\%',
         f'with relative errors below {max_rec_err:.2f}\\%'),
        # Conclusion: even at [TBD]% observational noise
        (r'even at \\textbf\{\[TBD\]\}\\% observational noise',
         f'even at 2.00\\% observational noise'),
        # Table 3 caption: within [TBD] percent
        (r'within \\textbf\{\[TBD\]\} percent',
         f'within {max_rec_err:.1f}\\% '),
    ]
    n_subs = 0
    for pat, repl in replacements:
        text, n = re.subn(pat, repl, text)
        n_subs += n

    # Per-snapshot table 2 cells: replace 8 cells in order with t=1, 5, 10, 20 vs anal/FD
    snap_pairs = [(per_snapshot[f't_{tt}']['L2_anal_pct'], per_snapshot[f't_{tt}']['L2_fd_pct'])
                  for tt in (1, 5, 10, 20)]
    snap_iter = iter([f'{v:.3f}' for pair in snap_pairs for v in pair])
    def _next_snap(m):
        try: return next(snap_iter) + '\\%'
        except StopIteration: return m.group(0)
    # Only substitute table cells in the rows starting with $t = N$
    def _row_sub(m):
        prefix = m.group(1)
        cells = m.group(2)
        new = re.sub(r'\\textbf\{\[TBD\]\}\\%', _next_snap, cells)
        return prefix + new
    text = re.sub(r'(\$t = \d+\$\s+&\s+)((?:\\textbf\{\[TBD\]\}\\%\s*&?\s*)+\\\\)',
                  _row_sub, text)

    # Table 3 row: 2% & [TBD] & [TBD]% & [TBD] & [TBD]%
    text = re.sub(
        r'\$2\\%\$\s+&\s+\\textbf\{\[TBD\]\}\s+&\s+\\textbf\{\[TBD\]\}\\%'
        r'\s+&\s+\\textbf\{\[TBD\]\}\s+&\s+\\textbf\{\[TBD\]\}\\%',
        f'$2\\%$    & {D_rec:.4f} & {rel_D:.2f}\\% & {k_rec:.4f} & {rel_k:.2f}\\%',
        text,
    )

    out_name = tex_name.replace('.tex', '_filled.tex')
    Path(out_name).write_text(text)
    remaining = len(re.findall(r'\\textbf\{\[TBD\]\}', text))
    print(f'Substitutions applied. [TBD] markers remaining: {remaining}')
    print(f'Saved {out_name}')
    try:
        from google.colab import files as colab_files
        colab_files.download(out_name)
    except ImportError:
        print('(Not on Colab - filled .tex remains in working directory.)')
else:
    print('Skipping auto-fill (set AUTO_FILL=True to enable).')

## 7. Download all artifacts

In [ ]:
import zipfile

archive = 'dopamine_PINN_2D_artifacts.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(FIG_DIR.iterdir()):
        if p.is_file():
            zf.write(p, arcname=p.name)
            print(f'  + {p.name}  ({p.stat().st_size // 1024} KB)')
print(f'\nWrote {archive} ({Path(archive).stat().st_size // 1024} KB)')

try:
    from google.colab import files as colab_files
    colab_files.download(archive)
except ImportError:
    print('(Not on Colab - archive remains in working directory.)')